# Agentic AI — TV show catalog

This notebook follows the **Agentic AI** lab style (`aisuite` tools + deterministic Pandas layer): **numbers come from Python code**; the LLM **narrates and packages** from precomputed JSON.

**Scenario:** synthetic show catalog by **genre** (Drama, Comedy), **0–10** scores (internal panel vs external buzz in snapshots).

In [ ]:
from pathlib import Path
import os
import sys

# Resolve project root (works from repo root or notebooks/ in Jupyter)
for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / "src" / "tv_catalog_tools.py").exists():
        PROJECT_ROOT = candidate.resolve()
        break
else:
    raise RuntimeError("Could not find src/tv_catalog_tools.py; open the notebook from the project tree.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)
print("PROJECT_ROOT:", PROJECT_ROOT)

## 1. Environment and aisuite client.

In [ ]:
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env")

import aisuite as ai

client = ai.Client()
MODEL = os.environ.get("AISUITE_DEFAULT_MODEL", "openai:gpt-4o-mini")
print("Model:", MODEL)

## 2. Synthetic data.

In [ ]:
import pandas as pd

shows = pd.read_csv(PROJECT_ROOT / "data" / "shows.csv")
panel = pd.read_csv(PROJECT_ROOT / "data" / "panel_ratings.csv")
display(shows.head())
display(panel.head())

## 3. Deterministic tools.

In [ ]:
import tv_catalog_tools as tools

GENRE = "Drama"
NUM_SEASONS = 2

print(tools.load_catalog_slice(GENRE))
print(tools.compute_genre_panel_stats(GENRE))
print(tools.estimate_recommendation_band(GENRE, NUM_SEASONS))

In [ ]:
from IPython.display import HTML

HTML(tools.build_comp_table_html(GENRE, NUM_SEASONS))

## 4. Agent with tool calling.

In [ ]:
prompt = (
    f"Summarize in English the recommendation scenario for genre {GENRE}, shows with {NUM_SEASONS} season(s). "
    "Use the tools to obtain facts; do not invent values."
)

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": prompt}],
    tools=[
        tools.load_catalog_slice,
        tools.compute_genre_panel_stats,
        tools.estimate_recommendation_band,
        tools.build_comp_table_html,
    ],
    max_turns=10,
)

print(response.choices[0].message.content)

## 5. Visualization.

In [ ]:
import matplotlib.pyplot as plt

import tv_catalog_utils as utils

fig = utils.plot_genre_scores(GENRE)
plt.show()

## 6. Full pipeline.

In [ ]:
from IPython.display import Markdown, display

out = utils.run_recommendation_pipeline(
    client,
    MODEL,
    genre=GENRE,
    num_seasons=NUM_SEASONS,
)

display(Markdown("### Tool-using analyst"))
print(out["analyst_tool_transcript"][:2000])

display(Markdown("### Executive narrative"))
display(Markdown(out["executive_narrative"]))

display(Markdown("### Reflection gate"))
print(out["reflection_gate"])

display(HTML(out["metrics"]["comp_table_html"]))